#**스마트폰 센서 데이터 기반 모션 분류**
# 단계3 : 단계별 모델링


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0.미션

단계별로 나눠서 모델링을 수행하고자 합니다.  

* 단계1 : 정적(0), 동적(1) 행동 분류 모델 생성
* 단계2 : 세부 동작에 대한 분류모델 생성
    * 단계1 모델에서 0으로 예측 -> 정적 행동 3가지 분류 모델링
    * 단계1 모델에서 1으로 예측 -> 동적 행동 3가지 분류 모델링
* 모델 통합
    * 두 단계 모델을 통합하고, 새로운 데이터에 대해서 최종 예측결과와 성능평가가 나오도록 함수로 만들기
* 성능 비교
    * 기본 모델링의 성능과 비교
    * 모든 모델링은 [다양한 알고리즘 + 성능 튜닝]을 수행해야 합니다.


## 1.환경설정

### (1) 라이브러리 불러오기

* 세부 요구사항
    - 기본적으로 필요한 라이브러리를 import 하도록 코드가 작성되어 있습니다.
    - 필요하다고 판단되는 라이브러리를 추가하세요.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 필요하다고 판단되는 라이브러리를 추가하세요.
from joblib import dump, load

import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import *

### (2) 데이터 불러오기

* 주어진 데이터셋
    * data01_train.csv : 학습 및 검증용

 <br/>  

* 세부 요구사항
    - data01_train.csv 를 불러와 'data' 이름으로 저장합니다.
        - data에서 변수 subject는 삭제합니다.
    - data01_test.csv 를 불러와 'new_data' 이름으로 저장합니다.


In [5]:
# 파일 경로 지정
data_file = r'/content/drive/MyDrive/실습파일/data01_train.csv'
data = pd.read_csv(data_file)
data.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,21,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,15,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,11,STANDING
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,17,WALKING
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,17,WALKING_DOWNSTAIRS


In [6]:
# 파일 불러오기
data_test_file = r'/content/drive/MyDrive/실습파일/data01_test.csv'
new_data = pd.read_csv(data_test_file)
data.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,21,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,15,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,11,STANDING
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,17,WALKING
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,17,WALKING_DOWNSTAIRS


In [7]:
# LGBM을 위한 JSON 데이터 처리
import re

data.columns = [re.sub(r'[^\w]', '_', col) for col in data.columns]

## 2.데이터 전처리

* 세부 요구사항
    - Label 추가 : data 에 Activity_dynamic 를 추가합니다. Activity_dynamic은 과제1에서 is_dynamic과 동일한 값입니다.
    - x와 y1, y2로 분할하시오.
        * y1 : Activity
        * y2 : Activity_dynamic
    - train : val = 8 : 2 혹은 7 : 3
    - random_state 옵션을 사용하여 다른 모델과 비교를 위해 성능이 재현되도록 합니다.

In [8]:
data['Activity_dynamic'] = data['Activity'].map({'WALKING':1,'WALKING_UPSTAIRS':1,'WALKING_DOWNSTAIRS':1,'LAYING':0,'STANDING':0,'SITTING':0})
data.head()

,tBodyAcc_mean___X,tBodyAcc_mean___Y,tBodyAcc_mean___Z,tBodyAcc_std___X,tBodyAcc_std___Y,tBodyAcc_std___Z,tBodyAcc_mad___X,tBodyAcc_mad___Y,tBodyAcc_mad___Z,tBodyAcc_max___X,...,angle_tBodyAccMean_gravity_,angle_tBodyAccJerkMean__gravityMean_,angle_tBodyGyroMean_gravityMean_,angle_tBodyGyroJerkMean_gravityMean_,angle_X_gravityMean_,angle_Y_gravityMean_,angle_Z_gravityMean_,subject,Activity,Activity_dynamic
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,21,STANDING,0
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,15,LAYING,0
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,11,STANDING,0
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,17,WALKING,1
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,17,WALKING_DOWNSTAIRS,1


In [9]:
X = data.drop(['Activity', 'Activity_dynamic'], axis = 1)
y1 = data['Activity']
y2 = data['Activity_dynamic']

In [10]:
sc = StandardScaler()
X = pd.DataFrame(sc.fit_transform(X), columns = X.columns)
X.head()

,tBodyAcc_mean___X,tBodyAcc_mean___Y,tBodyAcc_mean___Z,tBodyAcc_std___X,tBodyAcc_std___Y,tBodyAcc_std___Z,tBodyAcc_mad___X,tBodyAcc_mad___Y,tBodyAcc_mad___Z,tBodyAcc_max___X,...,fBodyBodyGyroJerkMag_skewness__,fBodyBodyGyroJerkMag_kurtosis__,angle_tBodyAccMean_gravity_,angle_tBodyAccJerkMean__gravityMean_,angle_tBodyGyroMean_gravityMean_,angle_tBodyGyroJerkMean_gravityMean_,angle_X_gravityMean_,angle_Y_gravityMean_,angle_Z_gravityMean_,subject
0,0.202596,0.218248,0.103371,-0.859791,-0.902765,-0.870784,-0.850437,-0.900177,-0.872323,-0.851929,...,-0.563399,-0.622364,-0.150183,-0.096236,0.490897,0.167590,-0.215351,0.914853,0.789202,0.404857
1,-0.133922,0.031040,0.192441,-0.861051,-0.966218,-0.918744,-0.853240,-0.964650,-0.925369,-0.865856,...,0.210864,-0.225448,-0.210252,0.871377,-1.273317,1.633370,1.643741,-2.786559,-0.335244,-0.266467
2,0.057662,0.083408,0.011623,-0.879254,-0.939217,-0.934495,-0.871492,-0.940565,-0.943429,-0.875484,...,-0.710714,-0.663013,-0.024311,-1.171478,-1.481923,0.060053,-0.671996,0.481426,0.078430,-0.714017
3,0.221631,-0.449954,-0.701726,0.827624,1.003249,0.637317,0.835699,1.113624,0.623374,0.869399,...,0.935220,0.691463,-0.776120,1.373090,1.213607,-0.138128,-0.401392,0.766375,0.596080,-0.042693
4,1.774887,1.316574,3.437248,1.541168,0.803509,0.516570,1.455736,0.858146,0.627111,1.926091,...,0.458672,0.193691,-0.155629,-1.887657,-1.618035,-1.847602,-0.419484,0.691700,0.690781,-0.042693


In [11]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5881 entries, 0 to 5880
Columns: 562 entries, tBodyAcc_mean___X to subject
dtypes: float64(562)
memory usage: 25.2 MB


In [12]:
X_train, X_val, y_train, y_val = train_test_split(X, y2, test_size = 0.2, random_state = 42, shuffle=False)
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4704 entries, 0 to 4703
Columns: 562 entries, tBodyAcc_mean___X to subject
dtypes: float64(562)
memory usage: 20.2 MB


## **3.단계별 모델링**

![](https://github.com/DA4BAM/image/blob/main/step%20by%20step.png?raw=true)

### (1) 단계1 : 정적/동적 행동 분류 모델

* 세부 요구사항
    * 정적 행동(Laying, Sitting, Standing)과 동적 행동(동적 : Walking, Walking-Up, Walking-Down)을 구분하는 모델 생성.
    * 몇가지 모델을 만들고 가장 성능이 좋은 모델을 선정하시오.

#### 1) 알고리즘1 : XGBClassifier

In [13]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(X_train, y_train)

pred = xgb_model.predict(X_val)

#평가
print('accuracy :',accuracy_score(y_val, pred))
print('='*60)
print(confusion_matrix(y_val, pred))
print('='*60)
print(classification_report(y_val, pred))

# F1 스코어 확인 (micro, macro, weighted 등 선택 가능)
f1_xgb = f1_score(y_val, pred, average='weighted')
print('F1 Score (weighted):', f1_xgb)

accuracy : 1.0
[[658   0]
 [  0 519]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       658
           1       1.00      1.00      1.00       519

    accuracy                           1.00      1177
   macro avg       1.00      1.00      1.00      1177
weighted avg       1.00      1.00      1.00      1177

F1 Score (weighted): 1.0


#### 2) 알고리즘2 : RandomForestClassifier

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

pred = rf_model.predict(X_val)
print('accuracy :', accuracy_score(y_val, pred))
print('='*60)
print('Confusion Matrix:')
print(confusion_matrix(y_val, pred))
print('='*60)
print('Classification Report:')

# F1 스코어 확인 (weighted 평균)
f1_rf = f1_score(y_val, pred, average='weighted')
print('F1 Score (weighted):', f1_rf)

accuracy : 0.9991503823279524
Confusion Matrix:
[[657   1]
 [  0 519]]
Classification Report:
F1 Score (weighted): 0.9991504681425801


In [15]:
dump(xgb_model, 'model1_xgb.joblib')

['model1_xgb.joblib']

### (2) 단계2-1 : 정적 동작 세부 분류

* 세부 요구사항
    * 정적 행동(Laying, Sitting, Standing)인 데이터 추출
    * Laying, Sitting, Standing 를 분류하는 모델을 생성
    * 몇가지 모델을 만들고 가장 성능이 좋은 모델을 선정하시오.

In [16]:
data_static = data[data['Activity_dynamic'] == 0]
data_static.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3234 entries, 0 to 5880
Columns: 564 entries, tBodyAcc_mean___X to Activity_dynamic
dtypes: float64(561), int64(2), object(1)
memory usage: 13.9+ MB


In [17]:
X_static = data_static.drop(['Activity', 'Activity_dynamic'], axis = 1)
y1_static = data_static['Activity'].map({'LAYING':0, 'STANDING':1, 'SITTING':2})

In [18]:
X_static = pd.DataFrame(sc.transform(X_static), columns = X_static.columns)
X_static.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3234 entries, 0 to 3233
Columns: 562 entries, tBodyAcc_mean___X to subject
dtypes: float64(562)
memory usage: 13.9 MB


In [19]:
X_train, X_val, y_train, y_val = train_test_split(X_static, y1_static, test_size = 0.2, random_state = 42, shuffle=False)

### XGBClassfier

In [20]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(X_train, y_train)

pred = xgb_model.predict(X_val)

#평가
print('accuracy :',accuracy_score(y_val, pred))
print('='*60)
print(confusion_matrix(y_val, pred))
print('='*60)
print(classification_report(y_val, pred))

# F1 스코어 확인 (micro, macro, weighted 등 선택 가능)
f1_xgb = f1_score(y_val, pred, average='weighted')
print('F1 Score (weighted):', f1_xgb)

accuracy : 0.9922720247295209
[[229   0   0]
 [  0 214   3]
 [  0   2 199]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       229
           1       0.99      0.99      0.99       217
           2       0.99      0.99      0.99       201

    accuracy                           0.99       647
   macro avg       0.99      0.99      0.99       647
weighted avg       0.99      0.99      0.99       647

F1 Score (weighted): 0.9922726890291964


In [21]:
# HyperOpt 검색 공간 설정
from hyperopt import hp

xgb_search_space = {'max_depth':hp.quniform('max_depth', 5, 20, 1),
                   'min_child_weight':hp.quniform('min_child_weight', 1, 2, 1),
                   'learning_rate':hp.uniform('learning_rate', 0.01, 0.2),
                   'colsample_bytree':hp.uniform('colsample_bytree', 0.5, 1)}

In [22]:
# HyperOpt 목적 함수 설정
from sklearn.model_selection import cross_val_score # 교차검증
from hyperopt import STATUS_OK

def objective_func(search_space):
    xgb_clf = XGBClassifier(
        n_estimators=100,
        max_depth=int(search_space['max_depth']), # 정수형 하이퍼 파라미터 형변환 필요:int형
        min_child_weight=int(search_space['min_child_weight']), # 정수형 하이퍼 파라미터 형변환 필요:int형
        learning_rate=search_space['learning_rate'],
        colsample_bytree=search_space['colsample_bytree'],
        eval_metric='logloss')

    # 목적 함수의 반환값은 교차검증 기반의 평균 정확도 사용
    accuracy = cross_val_score(xgb_clf, X_train, y_train, scoring='accuracy', cv=3)

    # accuracy는 cv=3 개수만큼의 결과를 리스트로 가짐. 이를 평균하여 반환하되 -1을 곱함
    return {'loss':-1 * np.mean(accuracy), 'status':STATUS_OK}

In [23]:
# HyperOpt fmin()을 이용해 최적 하이퍼 파라미터 도출
from hyperopt import fmin, tpe, Trials

trial_val = Trials()

best = fmin(fn=objective_func,
           space=xgb_search_space,
           algo=tpe.suggest,
           max_evals=20, # 입력값 시도 횟수 지정
           trials=trial_val,
           # rstate=np.random.default_rng(seed=9)
           )

print('best:', best)

100%|██████████| 20/20 [20:47<00:00, 62.38s/trial, best loss: -0.974102462766353]
best: {'colsample_bytree': np.float64(0.9530974782421355), 'learning_rate': np.float64(0.11774177543821601), 'max_depth': np.float64(12.0), 'min_child_weight': np.float64(2.0)}


In [24]:
# 모델 선언
xgb_tunning_static = XGBClassifier(
    n_estimators=400,
    learning_rate=round(best['learning_rate'], 5),
    max_depth=int(best['max_depth']),
    min_child_weight=int(best['min_child_weight']),
    colsample_bytree=round(best['colsample_bytree'], 5)
)

## model train
xgb_tunning_static.fit(
    X_train, y_train,
    verbose=True
)

pred_tunning = xgb_tunning_static.predict(X_val)

#평가
print('accuracy :',accuracy_score(y_val, pred_tunning))
print('='*60)
print(confusion_matrix(y_val, pred_tunning))
print('='*60)
print(classification_report(y_val, pred_tunning))

# F1 스코어 확인 (micro, macro, weighted 등 선택 가능)
f1_xgb = f1_score(y_val, pred_tunning, average='weighted')
print('F1 Score (weighted):', f1_xgb)

accuracy : 0.9938176197836167
[[229   0   0]
 [  0 215   2]
 [  0   2 199]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       229
           1       0.99      0.99      0.99       217
           2       0.99      0.99      0.99       201

    accuracy                           0.99       647
   macro avg       0.99      0.99      0.99       647
weighted avg       0.99      0.99      0.99       647

F1 Score (weighted): 0.9938176197836167


### RandomForest

In [25]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

pred = rf_model.predict(X_val)
print('accuracy :', accuracy_score(y_val, pred))
print('='*60)
print('Confusion Matrix:')
print(confusion_matrix(y_val, pred))
print('='*60)
print('Classification Report:')

# F1 스코어 확인 (weighted 평균)
f1_rf = f1_score(y_val, pred, average='weighted')
print('F1 Score (weighted):', f1_rf)

accuracy : 0.9752704791344667
Confusion Matrix:
[[229   0   0]
 [  0 215   2]
 [  0  14 187]]
Classification Report:
F1 Score (weighted): 0.9752227091045091


In [26]:
# HyperOpt 검색 공간 설정
from hyperopt import hp

rf_search_space = {
    'n_estimators': hp.quniform('n_estimators', 100, 500, 50),
    'max_depth': hp.quniform('max_depth', 5, 30, 1),
    'min_samples_split': hp.quniform('min_samples_split', 2, 10, 1),
    'min_samples_leaf': hp.quniform('min_samples_leaf', 1, 5, 1),
    'max_features': hp.choice('max_features', ['sqrt', 'log2', None])
}

In [27]:
# HyperOpt 목적 함수 설정
from sklearn.model_selection import cross_val_score
from hyperopt import STATUS_OK

def rf_objective_func(search_space):
    rf_clf = RandomForestClassifier(
        n_estimators=int(search_space['n_estimators']),
        max_depth=int(search_space['max_depth']),
        min_samples_split=int(search_space['min_samples_split']),
        min_samples_leaf=int(search_space['min_samples_leaf']),
        max_features=search_space['max_features'],
        random_state=42
    )

    accuracy = cross_val_score(rf_clf, X_train, y_train, scoring='accuracy', cv=3)
    return {'loss': -1 * np.mean(accuracy), 'status': STATUS_OK}

In [28]:
# HyperOpt fmin()을 이용해 최적 하이퍼 파라미터 도출
from hyperopt import fmin, tpe, Trials

rf_trials = Trials()

best_rf = fmin(
    fn=rf_objective_func,
    space=rf_search_space,
    algo=tpe.suggest,
    max_evals=20,
    trials=rf_trials
)

print('Best RF Hyperparameters:', best_rf)

100%|██████████| 20/20 [1:19:38<00:00, 238.90s/trial, best loss: -0.9671423540070924]
Best RF Hyperparameters: {'max_depth': np.float64(27.0), 'max_features': np.int64(1), 'min_samples_leaf': np.float64(3.0), 'min_samples_split': np.float64(6.0), 'n_estimators': np.float64(450.0)}


In [29]:
# max_features는 choice라서 index로 반환됨 → 다시 값으로 매핑
max_features_options = ['sqrt', 'log2', None]
best_rf['max_features'] = max_features_options[best_rf['max_features']]

rf_tuned = RandomForestClassifier(
    n_estimators=int(best_rf['n_estimators']),
    max_depth=int(best_rf['max_depth']),
    min_samples_split=int(best_rf['min_samples_split']),
    min_samples_leaf=int(best_rf['min_samples_leaf']),
    max_features=best_rf['max_features'],
    random_state=42
)

rf_tuned.fit(X_train, y_train)
pred_rf_tuned = rf_tuned.predict(X_val)

# 평가
print('accuracy :', accuracy_score(y_val, pred_rf_tuned))
print('='*60)
print(confusion_matrix(y_val, pred_rf_tuned))
print('='*60)
print(classification_report(y_val, pred_rf_tuned))

# F1 스코어 (weighted)
f1_rf_tuned = f1_score(y_val, pred_rf_tuned, average='weighted')
print('F1 Score (weighted):', f1_rf_tuned)

accuracy : 0.9752704791344667
[[229   0   0]
 [  0 211   6]
 [  0  10 191]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       229
           1       0.95      0.97      0.96       217
           2       0.97      0.95      0.96       201

    accuracy                           0.98       647
   macro avg       0.97      0.97      0.97       647
weighted avg       0.98      0.98      0.98       647

F1 Score (weighted): 0.9752591303719942


### SVM

In [30]:
from sklearn.svm import SVC

svm_model = SVC(random_state=42)
svm_model.fit(X_train, y_train)

pred = svm_model.predict(X_val)

# 평가
print('accuracy :', accuracy_score(y_val, pred))
print('='*60)
print(confusion_matrix(y_val, pred))
print('='*60)
print(classification_report(y_val, pred))

# F1 스코어 확인 (weighted 평균)
f1_svm = f1_score(y_val, pred, average='weighted')
print('F1 Score (weighted):', f1_svm)

accuracy : 0.9706336939721792
[[229   0   0]
 [  0 207  10]
 [  0   9 192]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       229
           1       0.96      0.95      0.96       217
           2       0.95      0.96      0.95       201

    accuracy                           0.97       647
   macro avg       0.97      0.97      0.97       647
weighted avg       0.97      0.97      0.97       647

F1 Score (weighted): 0.9706362183109459


### KNN

In [31]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier()
knn_model.fit(X_train, y_train)

pred = knn_model.predict(X_val)

# 평가
print('accuracy :', accuracy_score(y_val, pred))
print('='*60)
print(confusion_matrix(y_val, pred))
print('='*60)
print(classification_report(y_val, pred))

# F1 스코어 확인 (weighted 평균)
f1_knn = f1_score(y_val, pred, average='weighted')
print('F1 Score (weighted):', f1_knn)

accuracy : 0.9258114374034003
[[229   0   0]
 [  0 208   9]
 [  2  37 162]]
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       229
           1       0.85      0.96      0.90       217
           2       0.95      0.81      0.87       201

    accuracy                           0.93       647
   macro avg       0.93      0.92      0.92       647
weighted avg       0.93      0.93      0.92       647

F1 Score (weighted): 0.9249811489166284


#### SVM, KNN 은 정확도 낮아서 하이퍼 파라미터 적용 안 함

In [32]:
dump(xgb_tunning_static, 'model2_1_xgb.joblib')

['model2_1_xgb.joblib']

### (3) 단계2-2 : 동적 동작 세부 분류

* 세부 요구사항
    * 동동적 행동(Walking, Walking Upstairs, Walking Downstairs)인 데이터 추출
    * Walking, Walking Upstairs, Walking Downstairs 를 분류하는 모델을 생성
    * 몇가지 모델을 만들고 가장 성능이 좋은 모델을 선정하시오.

In [33]:
data_dynamic = data[data['Activity_dynamic'] == 1]
data_dynamic.head()

,tBodyAcc_mean___X,tBodyAcc_mean___Y,tBodyAcc_mean___Z,tBodyAcc_std___X,tBodyAcc_std___Y,tBodyAcc_std___Z,tBodyAcc_mad___X,tBodyAcc_mad___Y,tBodyAcc_mad___Z,tBodyAcc_max___X,...,angle_tBodyAccMean_gravity_,angle_tBodyAccJerkMean__gravityMean_,angle_tBodyGyroMean_gravityMean_,angle_tBodyGyroJerkMean_gravityMean_,angle_X_gravityMean_,angle_Y_gravityMean_,angle_Z_gravityMean_,subject,Activity,Activity_dynamic
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,17,WALKING,1
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,17,WALKING_DOWNSTAIRS,1
5,0.330708,0.007561,-0.061371,-0.215760,0.101075,0.072949,-0.269857,0.060060,0.101298,-0.019263,...,-0.030645,-0.852091,-0.500195,0.306091,-0.552729,0.253885,0.291256,23,WALKING_UPSTAIRS,1
6,0.121465,-0.031902,-0.005196,-0.152198,-0.113104,-0.239423,-0.202401,-0.164698,-0.247099,0.114668,...,0.445206,-0.003487,-0.940185,0.041387,-0.886603,0.173338,-0.005627,29,WALKING,1
12,0.303885,0.002768,-0.038613,-0.168656,0.190336,-0.140473,-0.205134,0.101144,-0.120572,-0.000818,...,-0.040030,0.257252,0.076091,-0.123425,-0.752882,0.266729,0.045692,1,WALKING,1


In [34]:
X_dynamic = data_dynamic.drop(['Activity', 'Activity_dynamic'], axis = 1)
y1_dynamic = data_dynamic['Activity'].map({'WALKING':0, 'WALKING_UPSTAIRS':1, 'WALKING_DOWNSTAIRS':2})

In [118]:
from sklearn.preprocessing import StandardScaler

X_train = data.drop('Activity', axis=1)
sc = StandardScaler()
sc.fit(X_dynamic)  # 훈련 데이터 기준으로 평균/표준편차 계산
X_dynamic = pd.DataFrame(sc.transform(X_dynamic), columns=X_dynamic.columns)

In [119]:
X_dynamic = pd.DataFrame(sc.transform(X_dynamic), columns = X_dynamic.columns)
y1_dynamic.head()

,Activity
3,0
4,2
5,1
6,0
12,0


In [120]:
X_train, X_val, y_train, y_val = train_test_split(X_dynamic, y1_dynamic, test_size = 0.2, random_state = 42, shuffle=False)

### XGB

In [38]:
from xgboost import XGBClassifier

xgb_model_dynamic = XGBClassifier(random_state=42)
xgb_model_dynamic.fit(X_train, y_train)

pred = xgb_model_dynamic.predict(X_val)

#평가
print('accuracy :',accuracy_score(y_val, pred))
print('='*60)
print(confusion_matrix(y_val, pred))
print('='*60)
print(classification_report(y_val, pred))

# F1 스코어 확인 (micro, macro, weighted 등 선택 가능)
f1_xgb = f1_score(y_val, pred, average='weighted')
print('F1 Score (weighted):', f1_xgb)

accuracy : 0.9962264150943396
[[191   2   0]
 [  0 179   0]
 [  0   0 158]]
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       193
           1       0.99      1.00      0.99       179
           2       1.00      1.00      1.00       158

    accuracy                           1.00       530
   macro avg       1.00      1.00      1.00       530
weighted avg       1.00      1.00      1.00       530

F1 Score (weighted): 0.996227070230608


In [ ]:
# HyperOpt 검색 공간 설정
from hyperopt import hp

xgb_search_space = {'max_depth':hp.quniform('max_depth', 5, 20, 1),
                   'min_child_weight':hp.quniform('min_child_weight', 1, 2, 1),
                   'learning_rate':hp.uniform('learning_rate', 0.01, 0.2),
                   'colsample_bytree':hp.uniform('colsample_bytree', 0.5, 1)}

In [ ]:
# HyperOpt 목적 함수 설정
from sklearn.model_selection import cross_val_score # 교차검증
from hyperopt import STATUS_OK

def objective_func(search_space):
    xgb_clf = XGBClassifier(
        n_estimators=100,
        max_depth=int(search_space['max_depth']), # 정수형 하이퍼 파라미터 형변환 필요:int형
        min_child_weight=int(search_space['min_child_weight']), # 정수형 하이퍼 파라미터 형변환 필요:int형
        learning_rate=search_space['learning_rate'],
        colsample_bytree=search_space['colsample_bytree'],
        eval_metric='logloss')

    # 목적 함수의 반환값은 교차검증 기반의 평균 정확도 사용
    accuracy = cross_val_score(xgb_clf, X_train, y_train, scoring='accuracy', cv=3)

    # accuracy는 cv=3 개수만큼의 결과를 리스트로 가짐. 이를 평균하여 반환하되 -1을 곱함
    return {'loss':-1 * np.mean(accuracy), 'status':STATUS_OK}

In [41]:
# HyperOpt fmin()을 이용해 최적 하이퍼 파라미터 도출
from hyperopt import fmin, tpe, Trials

trial_val = Trials()

best = fmin(fn=objective_func,
           space=xgb_search_space,
           algo=tpe.suggest,
           max_evals=20, # 입력값 시도 횟수 지정
           trials=trial_val,
           # rstate=np.random.default_rng(seed=9)
           )

print('best:', best)

100%|██████████| 20/20 [17:36<00:00, 52.81s/trial, best loss: -0.9905524414173681]
best: {'colsample_bytree': np.float64(0.5683418543262588), 'learning_rate': np.float64(0.1429554938410573), 'max_depth': np.float64(13.0), 'min_child_weight': np.float64(2.0)}


In [ ]:
# 모델 선언
xgb_tunning_dynamic = XGBClassifier(
    n_estimators=400,
    learning_rate=round(best['learning_rate'], 5),
    max_depth=int(best['max_depth']),
    min_child_weight=int(best['min_child_weight']),
    colsample_bytree=round(best['colsample_bytree'], 5)
)

## model train
xgb_tunning_dynamic.fit(
    X_train, y_train,
    verbose=True
)

pred_tunning = xgb_tunning_dynamic.predict(X_val)

#평가
print('accuracy :',accuracy_score(y_val, pred_tunning))
print('='*60)
print(confusion_matrix(y_val, pred_tunning))
print('='*60)
print(classification_report(y_val, pred_tunning))

# F1 스코어 확인 (micro, macro, weighted 등 선택 가능)
f1_xgb = f1_score(y_val, pred_tunning, average='weighted')
print('F1 Score (weighted):', f1_xgb)

In [ ]:
import pickle

#### 지금까지 한 하이퍼 저장 하는 셀

# best: 최적 하이퍼파라미터 딕셔너리
# trial_val: hyperopt의 Trials 객체

with open('/content/drive/MyDrive/trials_xgb.pkl', 'wb') as f:
    pickle.dump(trial_val, f)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score

# 모델 선언 및 학습
rf_model_dynamic = RandomForestClassifier(random_state=42)
rf_model_dynamic.fit(X_train, y_train)

# 예측
pred_dynamic = rf_model_dynamic.predict(X_val)

# 평가
print('accuracy :', accuracy_score(y_val, pred_dynamic))
print('='*60)
print(confusion_matrix(y_val, pred_dynamic))
print('='*60)
print(classification_report(y_val, pred_dynamic))

# F1 스코어 확인 (weighted 평균)
f1_rf = f1_score(y_val, pred_dynamic, average='weighted')
print('F1 Score (weighted):', f1_rf)

In [ ]:
from hyperopt import hp

rf_search_space = {
    'n_estimators': hp.quniform('n_estimators', 100, 500, 50),
    'max_depth': hp.quniform('max_depth', 5, 30, 1),
    'min_samples_split': hp.quniform('min_samples_split', 2, 10, 1),
    'min_samples_leaf': hp.quniform('min_samples_leaf', 1, 5, 1),
    'max_features': hp.choice('max_features', ['sqrt', 'log2', None])
}

In [ ]:
from sklearn.model_selection import cross_val_score
from hyperopt import STATUS_OK

def rf_objective_func(search_space):
    rf_clf = RandomForestClassifier(
        n_estimators=int(search_space['n_estimators']),
        max_depth=int(search_space['max_depth']),
        min_samples_split=int(search_space['min_samples_split']),
        min_samples_leaf=int(search_space['min_samples_leaf']),
        max_features=search_space['max_features'],
        random_state=42
    )

    accuracy = cross_val_score(rf_clf, X_train, y_train, scoring='accuracy', cv=3)
    return {'loss': -1 * np.mean(accuracy), 'status': STATUS_OK}

In [ ]:
from hyperopt import fmin, tpe, Trials

rf_trials = Trials()

best_rf = fmin(
    fn=rf_objective_func,
    space=rf_search_space,
    algo=tpe.suggest,
    max_evals=20,
    trials=rf_trials
)

print('best_rf:', best_rf)

In [ ]:
# max_features는 choice이므로 index → 실제 값으로 변환
max_features_options = ['sqrt', 'log2', None]
best_rf['max_features'] = max_features_options[best_rf['max_features']]

# 모델 선언
rf_tuning_dynamic = RandomForestClassifier(
    n_estimators=int(best_rf['n_estimators']),
    max_depth=int(best_rf['max_depth']),
    min_samples_split=int(best_rf['min_samples_split']),
    min_samples_leaf=int(best_rf['min_samples_leaf']),
    max_features=best_rf['max_features'],
    random_state=42
)

# 모델 학습
rf_tuning_dynamic.fit(X_train, y_train)

# 예측
pred_rf_tuning = rf_tuning_dynamic.predict(X_val)

# 평가
print('accuracy :', accuracy_score(y_val, pred_rf_tuning))
print('='*60)
print(confusion_matrix(y_val, pred_rf_tuning))
print('='*60)
print(classification_report(y_val, pred_rf_tuning))

# F1 스코어 확인 (weighted 평균)
f1_rf = f1_score(y_val, pred_rf_tuning, average='weighted')
print('F1 Score (weighted):', f1_rf)

In [60]:
dump(rf_model_dynamic, 'model2_2_rf.joblib')

['model2_2_rf.joblib']

In [51]:
from sklearn.ensemble import RandomForestClassifier

# 동적 데이터 준비
data_dynamic = data[data['Activity_dynamic'] == 1]
X_dynamic = data_dynamic.drop(['Activity', 'Activity_dynamic'], axis=1)
y1_dynamic = data_dynamic['Activity'].map({
    'WALKING': 0,
    'WALKING_UPSTAIRS': 1,
    'WALKING_DOWNSTAIRS': 2
})

# 스케일링 (기존 sc 사용)
X_dynamic_scaled = pd.DataFrame(sc.transform(X_dynamic), columns=X_dynamic.columns)

# 학습용/검증용 분리
from sklearn.model_selection import train_test_split
X_dyn_train, X_dyn_val, y_dyn_train, y_dyn_val = train_test_split(
    X_dynamic_scaled, y1_dynamic, test_size=0.2, random_state=42, shuffle=False
)

# 모델 학습
rf_dynamic = RandomForestClassifier(random_state=42)
rf_dynamic.fit(X_dyn_train, y_dyn_train)

# 성능 확인
from sklearn.metrics import classification_report, f1_score
pred = rf_dynamic.predict(X_dyn_val)
print(classification_report(y_dyn_val, pred))
print("F1 score (weighted):", f1_score(y_dyn_val, pred, average='weighted'))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99       193
           1       0.99      0.99      0.99       179
           2       0.98      0.99      0.99       158

    accuracy                           0.99       530
   macro avg       0.99      0.99      0.99       530
weighted avg       0.99      0.99      0.99       530

F1 score (weighted): 0.9905799439302243


### (4) 분류 모델 합치기


* 세부 요구사항
    * 두 단계 모델을 통합하고, 새로운 데이터(test)에 대해서 최종 예측결과와 성능평가가 나오도록 함수로 만들기
    * 데이터 파이프라인 구축 : test데이터가 로딩되어 전처리 과정을 거치고, 예측 및 성능 평가 수행

![](https://github.com/DA4BAM/image/blob/main/pipeline%20function.png?raw=true)

#### 1) 함수 만들기

In [122]:
# subject 컬럼이 없으면 불러오거나, 원래 csv로부터 다시 로드
new_data = pd.read_csv('/content/drive/MyDrive/실습파일/data01_test.csv')
new_data.columns = [re.sub(r'[^\w]', '_', col) for col in new_data.columns]  # 다시 정리
new_data.head()

,tBodyAcc_mean___X,tBodyAcc_mean___Y,tBodyAcc_mean___Z,tBodyAcc_std___X,tBodyAcc_std___Y,tBodyAcc_std___Z,tBodyAcc_mad___X,tBodyAcc_mad___Y,tBodyAcc_mad___Z,tBodyAcc_max___X,...,fBodyBodyGyroJerkMag_kurtosis__,angle_tBodyAccMean_gravity_,angle_tBodyAccJerkMean__gravityMean_,angle_tBodyGyroMean_gravityMean_,angle_tBodyGyroJerkMean_gravityMean_,angle_X_gravityMean_,angle_Y_gravityMean_,angle_Z_gravityMean_,subject,Activity
0,0.284379,-0.021981,-0.116683,-0.992490,-0.979640,-0.963321,-0.992563,-0.977304,-0.958142,-0.938850,...,-0.850065,-0.018043,0.092304,0.074220,-0.714534,-0.671943,-0.018351,-0.185733,22,SITTING
1,0.277440,-0.028086,-0.118412,-0.996620,-0.927676,-0.972294,-0.997346,-0.931405,-0.971788,-0.939837,...,-0.613367,-0.022456,-0.155414,0.247498,-0.112257,-0.826816,0.184489,-0.068699,15,STANDING
2,0.305833,-0.041023,-0.087303,0.006880,0.182800,-0.237984,0.005642,0.028616,-0.236474,0.016311,...,0.394388,-0.362616,0.171069,0.576349,-0.688314,-0.743234,0.272186,0.053101,22,WALKING
3,0.276053,-0.016487,-0.108381,-0.995379,-0.983978,-0.975854,-0.995877,-0.985280,-0.974907,-0.941425,...,-0.841455,0.289548,0.079801,-0.020033,0.291898,-0.639435,-0.111998,-0.123298,8,SITTING
4,0.271998,0.016904,-0.078856,-0.973468,-0.702462,-0.869450,-0.979810,-0.711601,-0.856807,-0.920760,...,0.214219,0.010111,0.114179,-0.830776,-0.325098,-0.840817,0.116237,-0.096615,5,STANDING


In [137]:
import re
from sklearn.metrics import accuracy_score, classification_report, f1_score

def data_pipeline(new_data, model1, model2_1, model2_2, sc, feature_columns):
    """
    두 단계 모델 예측 및 평가 파이프라인 함수

    Parameters:
        new_data (DataFrame): 테스트 데이터
        model1: 첫 번째 분류 모델
        model2_1: 두 번째 모델 (정적 동작 예측: Laying, Sitting, Standing)
        model2_2: 두 번째 모델 (동적 동작 예측: Walking, Walking_Up, Walking_Down)
        sc: 학습된 스케일러
        feature_columns: 모델 학습에 사용된 feature 컬럼 리스트

    Returns:
        dict: accuracy, classification_report 포함한 평가 결과
    """
    # 컬럼 이름 정리 (특수문자 → _)
    # subject 컬럼이 없으면 불러오거나, 원래 csv로부터 다시 로드
    new_data.columns = [re.sub(r'[^\w]', '_', col) for col in new_data.columns]  # 다시 정리

    # 정답 라벨 분리
    y_test = new_data['Activity']

    # feature_columns는 X_train.columns 기준이어야 함!
    X_test = new_data[feature_columns]  # 정확한 컬럼 순서로 정렬
    X_test_scaled = sc.transform(X_test)

    # 모델1 예측 (정적/동적 구분)
    model1_pred = model1.predict(X_test_scaled)

    # 모델2 예측
    final_preds = []
    for i, pred in enumerate(model1_pred):
        if pred == 0:  # 모델 2-1: LAYING, STANDING, SITTING
            row_df = pd.DataFrame([X_test_scaled[i]], columns=feature_columns)
            sub_pred = model2_1.predict(row_df)[0]
            activity_mapping = {0: 'LAYING', 1: 'STANDING', 2: 'SITTING'}
            final_preds.append(activity_mapping[sub_pred])
        else:  # 모델 2-2: WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS
            row_df = pd.DataFrame([X_test_scaled[i]], columns=feature_columns)
            sub_pred = model2_2.predict(row_df)[0]
            activity_mapping = {0: 'WALKING', 1: 'WALKING_UPSTAIRS', 2: 'WALKING_DOWNSTAIRS'}
            final_preds.append(activity_mapping[sub_pred])

    # 평가
    results = {
        'accuracy': accuracy_score(y_test, final_preds),
        'classification_report': classification_report(y_test, final_preds, output_dict=True)
    }

    print("Accuracy:", results['accuracy'])
    print("Classification Report:")
    print(classification_report(y_test, final_preds))
    print("F1 Score (weighted):", f1_score(y_test, final_preds, average='weighted'))

    return results

In [138]:
model1 = load('model1_xgb.joblib')
model2_1 = load('model2_1_xgb.joblib')
model2_2 = load('model2_2_rf.joblib')

In [139]:
feature_columns = list(X_train.columns)  # subject 포함된 562개

In [140]:
import pandas as pd
import re

new_data = pd.read_csv('/content/drive/MyDrive/실습파일/data01_test.csv')
new_data.columns = [re.sub(r'[^\w]', '_', col) for col in new_data.columns]  # 컬럼 이름 깨끗하게

In [141]:
print("model1 expects:", model1.n_features_in_)
print("new_data columns:", len(feature_columns), "==", len(new_data[feature_columns].columns))

model1 expects: 562
new_data columns: 562 == 562


In [142]:
results = data_pipeline(new_data, model1, model2_1, model2_2, sc, feature_columns)

Accuracy: 0.5710401087695445
Classification Report:
                    precision    recall  f1-score   support

            LAYING       0.74      1.00      0.85       292
           SITTING       0.35      0.79      0.48       254
          STANDING       0.48      0.24      0.32       287
           WALKING       0.72      0.34      0.46       228
WALKING_DOWNSTAIRS       0.99      0.66      0.79       195
  WALKING_UPSTAIRS       0.63      0.35      0.45       215

          accuracy                           0.57      1471
         macro avg       0.65      0.56      0.56      1471
      weighted avg       0.64      0.57      0.56      1471

F1 Score (weighted): 0.5551438633027331
